# 07 — Ragas Evaluation

**What this notebook does:**
- Creates a golden dataset of 10 question-answer pairs from your PDF
- Runs Ragas evaluation metrics: faithfulness, answer relevancy, context precision
- Saves results to `data/ragas_results.json`
- Produces a score you can quote in your resume and LinkedIn post

**What is Ragas?**
Ragas measures RAG quality without needing human judges:
- **Faithfulness**: are all claims in the answer supported by retrieved chunks?
- **Answer Relevancy**: does the answer actually address the question?
- **Context Precision**: were the right chunks retrieved (not irrelevant ones)?

In [1]:
import chromadb
import numpy as np
import json
import os
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection("my_docs")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0, api_key=GROQ_API_KEY)

all_data   = collection.get(include=["documents"])
all_chunks = all_data["documents"]
all_ids    = all_data["ids"]
tokenised  = [c.lower().split() for c in all_chunks]
bm25       = BM25Okapi(tokenised)

print("All components loaded")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


All components loaded


## Step 1 — Rebuild the production pipeline

In [2]:
def vector_search(query, top_k=10):
    vec = embed_model.encode([query]).tolist()
    res = collection.query(query_embeddings=vec, n_results=top_k)
    return res["ids"][0], res["documents"][0]

def bm25_search(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [all_ids[i] for i in top_idx], [all_chunks[i] for i in top_idx]

def rrf_fusion(v_ids, b_ids, k=60):
    scores = {}
    for rank, cid in enumerate(v_ids):
        scores[cid] = scores.get(cid, 0) + 1/(k+rank+1)
    for rank, cid in enumerate(b_ids):
        scores[cid] = scores.get(cid, 0) + 1/(k+rank+1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

SYSTEM_PROMPT = """You are DocMind, a precise document assistant.
Answer using ONLY the provided context. Be concise.
End with: Sources: [chunk_id]
If insufficient context, say: INSUFFICIENT_CONTEXT"""

def full_rag_pipeline(query):
    """Returns answer and retrieved contexts for Ragas evaluation."""
    v_ids, v_chunks = vector_search(query)
    b_ids, b_chunks = bm25_search(query)
    fused = rrf_fusion(v_ids, b_ids)
    id_to_chunk = {cid: c for cid, c in zip(v_ids+b_ids, v_chunks+b_chunks)}
    candidates = [{"id": cid, "text": id_to_chunk[cid]} for cid, _ in fused[:10] if cid in id_to_chunk]
    pairs = [(query, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)
    for i, c in enumerate(candidates):
        c["rerank_score"] = float(rerank_scores[i])
    top3 = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:3]

    context_str = "".join([f"[{r['id']}]\n{r['text']}\n" for r in top3])
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Context:\n{context_str}\n---\nQuestion: {query}\nAnswer:")
    ]
    response = llm.invoke(messages)
    return response.content, [r["text"] for r in top3]

print("Pipeline ready")

Pipeline ready


## Step 2 — Create your golden dataset

The ground truth answers should be factually correct based on your document.
Write at least 20 pairs for a meaningful score.

In [3]:
GOLDEN_DATASET = [
    {
        "question": "What is an operating system?",
        "ground_truth": "An operating system acts as an intermediary between the user of a computer and computer hardware. It manages resources and provides an environment to execute programs conveniently and efficiently."
    },
    {
        "question": "What are the four necessary conditions for deadlock?",
        "ground_truth": "The four necessary conditions for deadlock are mutual exclusion, hold and wait, no preemption, and circular wait. All four must occur simultaneously for deadlock to happen."
    },
    {
        "question": "What is the Banker's algorithm?",
        "ground_truth": "The Banker's algorithm is a resource allocation and deadlock avoidance algorithm that tests for safety by simulating allocation for maximum possible amounts of all resources, then makes a safe-state check before deciding whether allocation should continue."
    },
    {
        "question": "What is a process control block?",
        "ground_truth": "A process control block (PCB) contains information associated with a specific process including process state, program counter, CPU registers, CPU scheduling information, memory management information, accounting information, and I/O status information."
    },
    {
        "question": "What is Round Robin scheduling?",
        "ground_truth": "In Round Robin scheduling, each process gets a small unit of CPU time called a time quantum, usually 10 to 100 milliseconds. After this time elapses, the process is preempted and added to the end of the ready queue."
    },
    {
        "question": "What is a semaphore?",
        "ground_truth": "A semaphore is an integer variable for which only two atomic operations are defined — wait and signal. It is used to synchronize operations between processes and can be binary or counting."
    },
    {
        "question": "What is the critical section problem?",
        "ground_truth": "The critical section problem involves designing a protocol for cooperative processes to access shared resources without creating data inconsistencies. Any solution must satisfy mutual exclusion, progress, and bounded waiting."
    },
    {
        "question": "What is SJF scheduling?",
        "ground_truth": "Shortest Job First scheduling associates each process with the length of its next CPU burst and schedules the process with the shortest time first. SJF is optimal as it gives minimum average waiting time for a given set of processes."
    },
    {
        "question": "What is a microkernel?",
        "ground_truth": "A microkernel runs on privileged mode and provides low-level address space management and inter-process communication. OS services such as file system and CPU scheduler run on top of the microkernel with their own address spaces, providing protection between components."
    },
    {
        "question": "What is a batch operating system?",
        "ground_truth": "In a batch operating system, jobs are executed in batches. The system puts all jobs in a queue on a first come first serve basis and executes them one by one. Users collect their output after all jobs are executed."
    },
    {
        "question": "What is a real-time operating system?",
        "ground_truth": "A real-time operating system serves real-time systems where the time interval required to process and respond to inputs is very small. Each job carries a deadline within which it must be completed. There are two types: hard real-time and soft real-time systems."
    },
    {
        "question": "What is a thread?",
        "ground_truth": "A thread is a basic unit of CPU utilization comprising a thread ID, program counter, register set, and stack. It shares code section, data section, and other operating system resources with other threads belonging to the same process."
    },
    {
        "question": "What is FCFS scheduling?",
        "ground_truth": "First Come First Served scheduling executes processes in the order they arrive. The process that requests CPU first gets it first. It is non-preemptive and can suffer from the convoy effect where short processes wait behind long ones."
    },
    {
        "question": "What is a race condition?",
        "ground_truth": "A race condition occurs when more than one process accesses the same shared data or code concurrently and the outcome depends on the particular order in which access takes place, leading to incorrect or inconsistent results."
    },
    {
        "question": "What is virtual memory?",
        "ground_truth": "Virtual memory abstracts the hardware of a computer such as CPU, memory, and disk drives into multiple execution environments, giving the feel that each environment is a single computer."
    },
    {
        "question": "What is a distributed operating system?",
        "ground_truth": "A distributed operating system connects various autonomous interconnected computers that communicate using a shared network. Independent systems have their own memory and CPU and are referred to as loosely coupled systems."
    },
    {
        "question": "What is context switching?",
        "ground_truth": "When the CPU switches to another process, the system must save the state of the old process and load the saved state for the new process. Context switch time is overhead during which the system does no useful work."
    },
    {
        "question": "What is Peterson's solution?",
        "ground_truth": "Peterson's solution is a classical software-based solution to the critical section problem using two shared variables: a boolean flag array and an integer turn variable. It satisfies mutual exclusion, progress, and bounded waiting but is limited to two processes."
    },
    {
        "question": "What are the advantages of multithreading?",
        "ground_truth": "The benefits of multithreaded programming include responsiveness, resource sharing, economy since threads share process resources making creation cheaper than processes, and scalability as threads can run in parallel on multiple processor cores."
    },
    {
        "question": "What is a monolithic kernel?",
        "ground_truth": "In a monolithic kernel, the entire operating system runs as a single program in kernel mode. Any procedure can call any other procedure. All procedures are compiled into a single executable file and have access to all modules."
    },
]
print(f"Golden dataset size: {len(GOLDEN_DATASET)} pairs")
print("Remember: replace placeholders with real Q&A from your PDF!")

Golden dataset size: 20 pairs
Remember: replace placeholders with real Q&A from your PDF!


## Step 3 — Run the pipeline on all questions

In [4]:
print("Running pipeline on all questions...")
print("(This calls Groq API once per question — takes ~2 mins for 20 questions)\n")

eval_data = {
    "question":   [],
    "answer":     [],
    "contexts":   [],
    "ground_truth": []
}

for i, item in enumerate(GOLDEN_DATASET):
    q = item["question"]
    gt = item["ground_truth"]

    print(f"[{i+1}/{len(GOLDEN_DATASET)}] {q[:60]}...")

    try:
        answer, contexts = full_rag_pipeline(q)
        eval_data["question"].append(q)
        eval_data["answer"].append(answer)
        eval_data["contexts"].append(contexts)
        eval_data["ground_truth"].append(gt)
    except Exception as e:
        print(f"  Error: {e} — skipping")

print(f"\nDone. {len(eval_data['question'])} questions processed.")

Running pipeline on all questions...
(This calls Groq API once per question — takes ~2 mins for 20 questions)

[1/20] What is an operating system?...


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[2/20] What are the four necessary conditions for deadlock?...
[3/20] What is the Banker's algorithm?...
[4/20] What is a process control block?...
[5/20] What is Round Robin scheduling?...
[6/20] What is a semaphore?...
[7/20] What is the critical section problem?...
[8/20] What is SJF scheduling?...
[9/20] What is a microkernel?...
[10/20] What is a batch operating system?...
[11/20] What is a real-time operating system?...
[12/20] What is a thread?...
[13/20] What is FCFS scheduling?...
[14/20] What is a race condition?...
[15/20] What is virtual memory?...
[16/20] What is a distributed operating system?...
[17/20] What is context switching?...
[18/20] What is Peterson's solution?...
[19/20] What are the advantages of multithreading?...
[20/20] What is a monolithic kernel?...

Done. 20 questions processed.


## Step 4 — Run Ragas evaluation

In [5]:
print("Running pipeline on all questions...")
print("This calls Groq once per question — takes ~2 mins for 20 questions\n")

eval_data = {
    "question":    [],
    "answer":      [],
    "contexts":    [],
    "ground_truth": []
}

for i, item in enumerate(GOLDEN_DATASET):
    q  = item["question"]
    gt = item["ground_truth"]

    print(f"[{i+1}/{len(GOLDEN_DATASET)}] {q[:55]}...")

    try:
        answer, contexts = full_rag_pipeline(q)
        eval_data["question"].append(q)
        eval_data["answer"].append(answer)
        eval_data["contexts"].append(contexts)
        eval_data["ground_truth"].append(gt)
        print(f"  Answer: {answer[:80]}...")
    except Exception as e:
        print(f"  ERROR: {e} — skipping")

print(f"\nDone. {len(eval_data['question'])} questions processed.")

Running pipeline on all questions...
This calls Groq once per question — takes ~2 mins for 20 questions

[1/20] What is an operating system?...
  Answer: An operating system acts as an intermediary between the user and computer hardwa...
[2/20] What are the four necessary conditions for deadlock?...
  Answer: The four necessary conditions for deadlock are:

1. Mutual Exclusion
2. Hold and...
[3/20] What is the Banker's algorithm?...
  Answer: The Banker's algorithm is a resource allocation and deadlock avoidance algorithm...
[4/20] What is a process control block?...
  Answer: A process control block (PCB) is a data structure in the operating system that r...
[5/20] What is Round Robin scheduling?...
  Answer: Round Robin (RR) scheduling is a CPU scheduling algorithm where each process get...
[6/20] What is a semaphore?...
  Answer: A semaphore is an integer variable for which only two atomic operations are defi...
[7/20] What is the critical section problem?...
  Answer: The critical 

In [6]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import evaluate, RunConfig # Add RunConfig to your imports

# Wrap models for Ragas
ragas_llm = LangchainLLMWrapper(ChatGroq(
    model="llama-3.1-8b-instant",    #llama-3.3-70b-versatile
    temperature=0.0,
    api_key=os.getenv("GROQ_API_KEY")
))

ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

# Build dataset
dataset = Dataset.from_dict(eval_data)
# Configure a 'safe' run setting for Groq
# Lowering max_workers stops the TimeoutError caused by too many parallel calls
run_config = RunConfig(
    timeout=180,      # Increase time limit per request to 180 seconds
    max_workers=1,    # Process 1 job at a time to stay under Groq rate limits
    max_retries=10    # Retry failed jobs up to 10 times
)

print("Running Ragas evaluation...")
print("(Takes ~3-5 mins)\n")

results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config,  # Pass the config here
    raise_exceptions=False  # Recommended: prevents one timeout from killing the whole script
)

print("\n" + "="*50)
print(" RAGAS EVALUATION RESULTS")
print("="*50)
print(f" Faithfulness      : {results['faithfulness']:.4f}")
print(f" Answer Relevancy  : {results['answer_relevancy']:.4f}")
print(f" Context Precision : {results['context_precision']:.4f}")
avg = (results['faithfulness'] + results['answer_relevancy'] + results['context_precision']) / 3
print(f" Average Score     : {avg:.4f}")
print("="*50)

C:\Users\Nirmal Choyal\AppData\Local\Temp\ipykernel_19948\1103667462.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Running Ragas evaluation...
(Takes ~3-5 mins)



Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[37]: TimeoutError()
Exception raised in Job[28]: TimeoutError()
Exception raised in Job[49]: TimeoutError()



 RAGAS EVALUATION RESULTS
 Faithfulness      : 0.8714
 Answer Relevancy  : 0.8986
 Context Precision : 1.0000
 Average Score     : 0.9233


## Step 5 — Save results to disk

In [7]:
os.makedirs("../data", exist_ok=True)

output = {
    "summary": {
        "faithfulness":      round(results['faithfulness'], 4),
        "answer_relevancy":  round(results['answer_relevancy'], 4),
        "context_precision": round(results['context_precision'], 4),
        "average":           round(avg, 4),
        "dataset_size":      len(GOLDEN_DATASET),
        "model":             "llama3-8b-8192",
        "embedding_model":   "all-MiniLM-L6-v2",
        "reranker":          "ms-marco-MiniLM-L-6-v2"
    },
    "per_question": [
        {
            "question": eval_data["question"][i],
            "answer":   eval_data["answer"][i],
            "ground_truth": eval_data["ground_truth"][i]
        }
        for i in range(len(eval_data["question"]))
    ]
}

with open("../data/ragas_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Results saved to ../data/ragas_results.json")


Results saved to ../data/ragas_results.json


## Key observations
- **Faithfulness < 0.5**: your prompt is too loose — tighten the system prompt
- **Answer Relevancy < 0.5**: retrieved chunks are off — try better chunking strategy
- **Context Precision < 0.5**: retrieval is noisy — tune hybrid search weights